# Stage I. Routing the test pool

The gate splits the test pool using only the cached Stage I probabilities and the frozen
threshold tau = 0.2391715943813324 (rounding it to 0.2392 misroutes one Grade-0 image). Images
with p(DR) >= tau continue to Stage II/III; the rest leave the cascade with the final grade 0.
Ground truth is merged in for the manifest and the confusion matrix only.

The original images are hard-linked (same file system) or copied; Stage II has its own
1,024-pixel front-end, so the 768-pixel Stage I preprocessing is not passed on.

**Steps.**
1. Write `dr_from_test/`, `nodr_from_test/` and the routing manifest.
2. Verify the folders against the manifest and recompute the gate confusion matrix.
3. Tabulate what Stage II receives, by true grade.

## 0. Configuration

In [ ]:
from pathlib import Path

# Inputs
#   outputs/stage1/eval/test_preds_768.csv   gate probabilities (notebook 02)
#   data/test/test_grade.csv, data/test/test_image/
# Outputs
#   outputs/stage1_gate/dr_from_test/        19,154 images -> Stage II (read by stage2 notebooks)
#   outputs/stage1_gate/nodr_from_test/      39,535 images -> final grade 0
#   data/test/stage1_manifest.csv            one row per test image (read by cascade notebooks)
DATA_ROOT = Path("data")
TEST_CSV = DATA_ROOT / "test" / "test_grade.csv"
TEST_IMG_DIR = DATA_ROOT / "test" / "test_image"
PRED_CACHE = Path("outputs") / "stage1" / "eval" / "test_preds_768.csv"
GATE_DIR = Path("outputs") / "stage1_gate"
MANIFEST_CSV = DATA_ROOT / "test" / "stage1_manifest.csv"

LINK_MODE = "auto"     # "auto" (hard link on the same file system, else copy) | "hardlink" | "copy"
WIPE_FIRST = True      # empty both output folders before writing

import pandas as pd
from hierarchiretina.stage1 import gate
from hierarchiretina.stage1.config import GATE_THRESHOLD

## 1. Route

Manifest columns: `filename, stem, prob, pred_binary, routed_to, true_grade, gt_binary,
gate_outcome, cascade_grade, has_file`, sorted by descending probability. `cascade_grade` is 0
for blocked images and empty for images that continue.

In [ ]:
manifest = gate.route_test_pool(PRED_CACHE, TEST_CSV, TEST_IMG_DIR, GATE_DIR, MANIFEST_CSV,
                                threshold=GATE_THRESHOLD, link_mode=LINK_MODE,
                                wipe_first=WIPE_FIRST)
manifest.head()

## 2. Verification

Expected: 19,154 / 39,535 images; TP 14,409, TN 37,076, FP 4,745, FN 2,459; sensitivity 0.8542,
specificity 0.8865.

In [ ]:
check = gate.verify_split(manifest, GATE_DIR)
pd.Series(check).to_frame("value")

## 3. What Stage II receives

The Grade-0 row is the false-positive load passed downstream (LG-DRG has no Grade-0 output, so
these images become Grade 1-4 errors); the blocked counts of Grades 1-5 are the cases the
cascade loses for good.

In [ ]:
tab = manifest.groupby("true_grade").agg(total=("stem", "size"), passed=("pred_binary", "sum"))
tab["blocked"] = tab.total - tab.passed
tab["pass_rate"] = (tab.passed / tab.total).round(4)
tab